# Jev, an LLM, and confidence-based escalation

A scheduling router predicts an intent such as `BLOCK_SLOTS`. Our judges check
whether it **failed to interpret an indirect or underspecified request**.
`TRUE` means that specific failure is present. `FALSE` can include other errors.

We compare Jev, GPT-4.1-mini, and Jev with uncertain cases sent to the LLM.
The latest dataset has 300 cases. Six worked examples appear in the prompt, so
we score the remaining **294 cases: 20 TRUE and 274 FALSE**.
The reference labels are provisional; inspect disagreements before trusting a score.

## 1. Setup

Run the cells in order. Use a TypeSafe API key for Jev and an OpenAI key for the LLM.
Keys come from `.env`, Colab Secrets, or hidden input. `auto` reuses a completed
local run; a fresh checkout makes API calls. [Provider and replay options](../README.md).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab needs a checkout; locally, open this notebook inside the repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    checkout = Path("/content/routing-evals-lab")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "codex/jev-routing-judge",
                        "https://github.com/The-Gen-Academy/Mastering-Agentic-AI-Week4-Session1.git",
                        str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "python-dotenv"], check=True)

folders = [Path.cwd(), Path("routing_agent_evals"), Path("AI-Evals-Principles/routing_agent_evals")]
LAB = next(folder.resolve() for folder in folders if (folder / "judge_lab.py").exists())
os.chdir(LAB)
sys.path.insert(0, str(LAB))

In [ ]:
import pandas as pd
from IPython.display import display
from prompts.llm_as_judge_prompt import (
    JEV_JUDGE_PROMPT, INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT,
)
from judge_lab import (
    load_dataset, batch, needs_llm, combined, metrics, always_false, confidence_bands,
)
from notebook_utils import setup_run, save_run

# After editing the prompt file, restart the kernel and run all cells.
client, RUN, manifest = setup_run(LAB, mode="auto", prompts={
    "jev": JEV_JUDGE_PROMPT,
    "llm": INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT,
})

## 2. Load the cases and prompts

Both judges receive only the doctor's request and the router's predicted intent.
Human labels are used afterward for scoring. The shared rules and six examples
live in [the prompt file](prompts/llm_as_judge_prompt.py), imported above.

In [ ]:
rows = load_dataset(LAB / "data")
data = pd.DataFrame(rows)
print(f"{len(data)} cases: {data.human_label.eq('TRUE').sum()} selected failures, "
      f"{data.human_label.eq('FALSE').sum()} other cases.")
display(data.head(3))

## 3. Run Jev

Jev uses **Choice** with `TRUE` and `FALSE`. Its response includes a label and confidence.
Confidence is not measured accuracy.

In [ ]:
jev = batch(rows, lambda row: client.call(row, "jev"))
example_id = rows[0]["id"]
display(pd.DataFrame([jev[example_id]]).reindex(columns=["label", "confidence", "probabilities", "error"]))

## 4. Run the LLM independently

GPT-4.1-mini sees the same rules and examples, without Jev's answer.
It returns a Boolean label and a short explanation; the loader converts the
Boolean to `TRUE` or `FALSE` for comparison.

In [ ]:
llm = batch(rows, lambda row: client.call(row, "llm"))
display(pd.DataFrame([llm[example_id]]).reindex(columns=["label", "reason", "error"]))

## 5. Keep Jev when confident; otherwise use the LLM

Keep a valid Jev answer at confidence **0.7 or above**. Otherwise use the LLM.
For binary Choice, probabilities 0.85 and 0.15 give confidence 0.70.
If the LLM fails too, leave the case unresolved.

We already called both models, so this loop reuses their answers. A standalone
cascade would call the LLM only on the escalated cases. [Confidence definition](https://docs.typesafe.ai/confidence).

In [ ]:
threshold = 0.7
cascade = {}
for row in rows:
    case_id = row["id"]
    if needs_llm(jev[case_id], threshold):
        cascade[case_id] = combined(jev[case_id], llm[case_id])
    else:
        cascade[case_id] = combined(jev[case_id])

print(f"LLM used for {sum(r['route'] == 'LLM' for r in cascade.values())}/{len(rows)} cases.")

## 6. Compare results

Check failure recall and false alarms alongside agreement. Always answering
`FALSE` scores highly on this imbalanced dataset while finding no failures.
Costs use reported charges or dated token-price estimates. Cascade cost counts
only Jev calls and escalated LLM calls; the full experiment calls both on every case.

In [ ]:
approaches = {"Always FALSE": always_false(rows), "Jev": jev, "LLM": llm, "Jev + LLM": cascade}
comparison = pd.DataFrame([
    metrics(rows, answers, name) for name, answers in approaches.items()
])
comparison["cost_per_1000_USD"] = comparison.cost_usd / len(rows) * 1000
comparison["LLM_fraction"] = comparison.llm_routes / len(rows)
comparison.loc[comparison.method.eq("LLM"), "LLM_fraction"] = 1.0
columns = ["method", "accuracy", "precision_TRUE", "recall_TRUE", "f1_TRUE",
           "FP", "unresolved", "LLM_fraction", "cost_per_1000_USD"]
table = comparison[columns].copy()
table.loc[comparison.unknown_cost_attempts.gt(0), "cost_per_1000_USD"] = float("nan")
display(table.round(4))

## 7. Inspect disagreements and save

The LLM helps only if it fixes more errors than it introduces on escalated cases.
Review the request, prediction, explanation, and reference label together.
These are practice results, not proof of performance on new traffic.

In [ ]:
bands = confidence_bands(rows, jev, llm, threshold)
display(pd.DataFrame(bands))

review = pd.DataFrame([{
    "ID": row["id"], "request": row["user_query"], "predicted_intent": row["predicted_intent"],
    "human": row["human_label"], "Jev": jev[row["id"]]["label"],
    "LLM": llm[row["id"]]["label"], "final": cascade[row["id"]]["label"],
    "LLM_reason": llm[row["id"]].get("reason"),
} for row in rows])
disagreements = review[review.Jev.ne(review.LLM) | review.final.ne(review.human)]
display(disagreements.head(8))
review.to_csv(RUN / "all_results.csv", index=False)
disagreements.to_csv(RUN / "disagreements.csv", index=False)
save_run(client, RUN, manifest, rows, threshold, comparison, bands)